# Clase 063 — Métricas: confusion matrix, precision, recall, F1

Dejamos de mirar la `accuracy` como métrica única. Aprendemos a leer una **matriz de confusión**, a elegir entre **precision**, **recall**, **F1** y **F-beta** según el costo de los errores, y a interpretar `classification_report`.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, fbeta_score, classification_report,
                             ConfusionMatrixDisplay)
from sklearn.datasets import load_digits, make_classification
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(42)

## 1. Confusión a mano vs `sklearn`

Con un caso chico verificamos que entendemos TP/FP/TN/FN. En sklearn las **filas son la clase real** y las **columnas la predicha**: `[[TN, FP], [FN, TP]]`.

In [ ]:
y_true = np.array([0, 1, 1, 0, 1, 1, 0, 0, 1, 0])
y_pred = np.array([0, 1, 0, 0, 1, 1, 1, 0, 1, 0])

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
print('matriz de confusion:\n', cm)
print(f'TP={tp} FP={fp} TN={tn} FN={fn}')

prec = tp / (tp + fp)
rec = tp / (tp + fn)
f1 = 2 * prec * rec / (prec + rec)
print(f'precision a mano: {prec:.3f} | sklearn: {precision_score(y_true, y_pred):.3f}')
print(f'recall a mano:    {rec:.3f} | sklearn: {recall_score(y_true, y_pred):.3f}')
print(f'F1 a mano:        {f1:.3f} | sklearn: {f1_score(y_true, y_pred):.3f}')

assert np.isclose(prec, precision_score(y_true, y_pred))
assert np.isclose(rec, recall_score(y_true, y_pred))
assert np.isclose(f1, f1_score(y_true, y_pred))

## 2. F-beta: ponderar recall o precision

F1 trata FP y FN como igual de caros. **F-beta** con β>1 pondera más el recall; con β<1 pondera más la precision. Usamos un predictor **conservador** (predice pocos positivos, todos correctos): precision alta, recall bajo.

In [ ]:
# predictor conservador: solo marca positivo cuando esta muy seguro
y_cons = np.array([0, 1, 0, 0, 1, 0, 0, 0, 1, 0])
print(f'precision: {precision_score(y_true, y_cons):.3f} | recall: {recall_score(y_true, y_cons):.3f}')

f05 = fbeta_score(y_true, y_cons, beta=0.5)   # prioriza precision
f1c = f1_score(y_true, y_cons)
f2 = fbeta_score(y_true, y_cons, beta=2.0)    # prioriza recall
print(f'F0.5 (precision pesa mas): {f05:.3f}')
print(f'F1  (balanceada):          {f1c:.3f}')
print(f'F2  (recall pesa mas):     {f2:.3f}')

# recall < precision -> F2 (pondera recall) es el mas bajo; F0.5 el mas alto
assert f2 < f1c < f05

## 3. Detector de 5s: matriz de confusión

Entrenamos el `SGDClassifier` binario "es un 5" sobre dígitos y mostramos la matriz con `ConfusionMatrixDisplay`, más precision/recall/F1.

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
ytr5, yte5 = (ytr == 5), (yte == 5)

sgd = SGDClassifier(random_state=42)
sgd.fit(Xtr, ytr5)
pred5 = sgd.predict(Xte)

print(f'precision: {precision_score(yte5, pred5):.3f}')
print(f'recall:    {recall_score(yte5, pred5):.3f}')
print(f'F1:        {f1_score(yte5, pred5):.3f}')

fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_predictions(yte5, pred5, ax=ax, colorbar=False)
ax.set_title('Detector "es un 5"')
plt.tight_layout()
plt.show()

## 4. `classification_report` multiclase

Con las 10 clases usamos `LogisticRegression` e imprimimos el reporte. Distinguimos `macro avg` (cada clase pesa igual) de `weighted avg` (ponderado por soporte).

In [ ]:
logreg = LogisticRegression(max_iter=2000, random_state=42, n_jobs=1)
logreg.fit(Xtr, ytr)
pred_multi = logreg.predict(Xte)
print(classification_report(yte, pred_multi, digits=3))

rec_por_clase = recall_score(yte, pred_multi, average=None)
peor = int(np.argmin(rec_por_clase))
print(f'clase con peor recall: digito {peor} (recall={rec_por_clase[peor]:.3f})')

## 5. Class imbalance: `class_weight='balanced'`

Con un dataset 99/1, el modelo sin ajuste ignora la clase rara (recall bajo). `class_weight='balanced'` reajusta la pérdida y recupera recall de la minoría.

In [ ]:
Xi, yi = make_classification(n_samples=10000, weights=[0.99, 0.01],
                             n_informative=5, random_state=42)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.3,
                                              stratify=yi, random_state=42)

base = LogisticRegression(max_iter=1000, random_state=42, n_jobs=1).fit(Xi_tr, yi_tr)
bal = LogisticRegression(max_iter=1000, class_weight='balanced',
                         random_state=42, n_jobs=1).fit(Xi_tr, yi_tr)

rec_base = recall_score(yi_te, base.predict(Xi_te))
rec_bal = recall_score(yi_te, bal.predict(Xi_te))
print(f'recall minoria SIN balance: {rec_base:.3f}')
print(f'recall minoria CON balance: {rec_bal:.3f}')

assert rec_bal > rec_base

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['sin class_weight', "class_weight='balanced'"], [rec_base, rec_bal],
       color=['#888', '#3a7'])
ax.set_ylabel('recall de la clase minoritaria')
ax.set_ylim(0, 1.05)
ax.set_title('class_weight rescata la clase rara')
plt.tight_layout()
plt.show()

## Ejercicios

1. **Confusión a mano.** Verificá TP/FP/TN/FN del ejercicio 1 con lápiz y papel y contrastá con `sklearn.metrics`.
2. **Peor dígito.** Sobre el `classification_report` multiclase, identificá qué dígito tiene peor recall y mirá con qué otro se confunde en la matriz.
3. **F-beta según costo.** Elegí β para un detector de fraude (FN caro) y otro para un filtro de spam (FP caro). Justificá.
4. **SMOTE (opcional).** Con `imbalanced-learn`, armá un `Pipeline` con `SMOTE` + `LogisticRegression` y compará el F1 de la minoría contra `class_weight='balanced'`. Recordá que SMOTE solo debe aplicarse al train fold (dentro del Pipeline).

## Conclusiones

- Toda métrica binaria sale de **TP/FP/TN/FN**; en sklearn filas=real, columnas=predicho.
- **Precision** importa cuando el FP es caro; **recall** cuando el FN es caro; **F1** cuando no hay preferencia clara.
- **F-beta** generaliza F1 para costos asimétricos (β>1 recall, β<1 precision).
- En multiclase, `macro avg` trata todas las clases por igual; `weighted avg` pondera por soporte.
- Con desbalanceo, `class_weight='balanced'` es el primer remedio (gratis, sin tocar el dataset).